In [1]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import warnings
warnings.filterwarnings('ignore')

c:\Users\Buwaneka Fernando\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BEST_MODEL_PATH = "../models/roberta_checkpoint"

In [3]:
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_PATH)
model     = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL_PATH)
model     = model.to(device)
model.eval()

print(f"Model loaded from: {BEST_MODEL_PATH}")
print(f"Device: {device}")

Model loaded from: ../models/roberta_checkpoint
Device: cpu


In [4]:
# This is what the entire agent is built around
def classify_product(product_text, category="unknown", max_length=256):
    """
    Takes a product description and classifies it as
    System 1 (emotional/impulsive) or System 2 (rational/deliberate).

    Returns a dictionary with:
    - cognitive_mode    : "System1" or "System2"
    - label             : 1 or 0
    - confidence        : float 0.0–1.0
    - s1_probability    : float (probability of being System 1)
    - s2_probability    : float (probability of being System 2)
    - reasoning         : human-readable explanation
    """

    # Format input same way as training data
    input_text = f"PRODUCT: {product_text}. REVIEW: {category}"

    # Tokenize
    encoding = tokenizer(
        input_text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)

    # Get prediction
    with torch.no_grad():
        outputs = model(**encoding)
        probs   = torch.softmax(outputs.logits, dim=1)
        pred    = torch.argmax(probs, dim=1).item()

    s2_prob = round(probs[0][0].item(), 4)
    s1_prob = round(probs[0][1].item(), 4)
    confidence = max(s1_prob, s2_prob)

    # Build human-readable reasoning
    if pred == 1:
        mode = "System1"
        if confidence > 0.85:
            reason = "Strong emotional/impulsive purchase signal detected. Low price, hedonic category, or emotional language patterns found."
        elif confidence > 0.70:
            reason = "Moderate emotional purchase signal. Product likely triggers impulse buying behavior."
        else:
            reason = "Weak emotional signal. Product may have both rational and emotional purchase drivers."
    else:
        mode = "System2"
        if confidence > 0.85:
            reason = "Strong rational/deliberate purchase signal detected. High consideration product with technical specifications or high price tier."
        elif confidence > 0.70:
            reason = "Moderate rational purchase signal. Consumers likely research before buying."
        else:
            reason = "Weak rational signal. Product sits between impulse and deliberate purchase behavior."

    return {
        "cognitive_mode":  mode,
        "label":           pred,
        "confidence":      round(confidence, 4),
        "s1_probability":  s1_prob,
        "s2_probability":  s2_prob,
        "reasoning":       reason
    }


# test
test_cases = [
    ("Neutrogena hydrating face wash with hyaluronic acid",        "Beauty"),
    ("Sony WH-1000XM5 Noise Cancelling Headphones 30hr battery",   "Electronics"),
    ("Organic gummy bears, pack of 6, natural fruit flavors",       "Grocery"),
    ("Garmin Forerunner 955 Solar GPS Running Smartwatch",          "Sports"),
    ("Cozy plush teddy bear gift set for babies",                   "Baby"),
]

print("\n─ CLASSIFIER TEST RESULTS ")
for product, category in test_cases:
    result = classify_product(product, category)
    print(f"\nProduct  : {product[:60]}")
    print(f"Category : {category}")
    print(f"Mode     : {result['cognitive_mode']} "
          f"(confidence: {result['confidence']:.2%})")
    print(f"S1 prob  : {result['s1_probability']:.4f} | "
          f"S2 prob  : {result['s2_probability']:.4f}")
    print(f"Reason   : {result['reasoning'][:80]}...")


─ CLASSIFIER TEST RESULTS 

Product  : Neutrogena hydrating face wash with hyaluronic acid
Category : Beauty
Mode     : System1 (confidence: 99.93%)
S1 prob  : 0.9993 | S2 prob  : 0.0007
Reason   : Strong emotional/impulsive purchase signal detected. Low price, hedonic category...

Product  : Sony WH-1000XM5 Noise Cancelling Headphones 30hr battery
Category : Electronics
Mode     : System2 (confidence: 99.96%)
S1 prob  : 0.0004 | S2 prob  : 0.9996
Reason   : Strong rational/deliberate purchase signal detected. High consideration product ...

Product  : Organic gummy bears, pack of 6, natural fruit flavors
Category : Grocery
Mode     : System1 (confidence: 99.94%)
S1 prob  : 0.9994 | S2 prob  : 0.0006
Reason   : Strong emotional/impulsive purchase signal detected. Low price, hedonic category...

Product  : Garmin Forerunner 955 Solar GPS Running Smartwatch
Category : Sports
Mode     : System2 (confidence: 99.93%)
S1 prob  : 0.0007 | S2 prob  : 0.9993
Reason   : Strong rational/delibera

In [6]:
def build_emotional_prompt(product_text, category, confidence):
    """
    System 1 prompt — designed to generate content that
    triggers fast, emotional, gut-level responses.
    Grounded in: Kahneman System 1, Emotional Contagion Theory
    """

    # Adjust emotional intensity based on confidence score
    if confidence > 0.85:
        intensity = "very strong"
        instruction = "Use vivid sensory language, excitement, and desire. Make it irresistible."
    elif confidence > 0.70:
        intensity = "moderate"
        instruction = "Use warm, positive emotional language. Focus on how it makes the user feel."
    else:
        intensity = "subtle"
        instruction = "Use gentle emotional appeal. Balance feeling with some light product information."

    prompt = f"""You are an expert neuro-marketing copywriter specializing in System 1 \
emotional marketing. System 1 thinking is fast, intuitive, and emotion-driven.

PRODUCT: {product_text}
CATEGORY: {category}
EMOTIONAL INTENSITY NEEDED: {intensity}

Write a SHORT marketing message (2-3 sentences, max 60 words) that:
1. Triggers immediate emotional desire using sensory or emotional language
2. Creates a feeling of pleasure, excitement, or belonging
3. Uses simple, vivid words that bypass rational thinking
4. Does NOT mention technical specifications or rational justifications
5. {instruction}

Write ONLY the marketing copy. No labels, no explanations."""

    return prompt


def build_rational_prompt(product_text, category, confidence):
    """
    System 2 prompt — designed to generate content that
    supports careful, deliberate, logic-based decision making.
    Grounded in: Kahneman System 2, Elaboration Likelihood Model
    """

    if confidence > 0.85:
        depth = "highly detailed"
        instruction = "Include specific numbers, comparisons, and technical advantages."
    elif confidence > 0.70:
        depth = "moderately detailed"
        instruction = "Highlight key features and value proposition clearly."
    else:
        depth = "balanced"
        instruction = "Combine key facts with a light value statement."

    prompt = f"""You are an expert neuro-marketing copywriter specializing in System 2 \
rational marketing. System 2 thinking is slow, analytical, and evidence-based.

PRODUCT: {product_text}
CATEGORY: {category}
DETAIL LEVEL NEEDED: {depth}

Write a SHORT marketing message (2-3 sentences, max 60 words) that:
1. Presents clear, logical reasons to purchase
2. Highlights specific features, benefits, or value
3. Uses precise, factual language that supports informed decision-making
4. Addresses potential objections or comparisons implicitly
5. {instruction}

Write ONLY the marketing copy. No labels, no explanations."""

    return prompt


def build_hybrid_prompt(product_text, category, s1_prob, s2_prob):
    """
    For borderline products (confidence < 0.65) — blends both styles.
    This handles the ambiguous cases your classifier is less sure about.
    """

    dominant = "emotional" if s1_prob > s2_prob else "rational"
    blend_ratio = f"{int(s1_prob*100)}% emotional, {int(s2_prob*100)}% rational"

    prompt = f"""You are an expert neuro-marketing copywriter. This product appeals to \
both emotional and rational buyers.

PRODUCT: {product_text}
CATEGORY: {category}
BLEND RATIO: {blend_ratio} (dominant: {dominant})

Write a SHORT marketing message (2-3 sentences, max 60 words) that:
1. Opens with an emotional hook to capture attention
2. Follows with one clear rational reason to justify the purchase
3. Closes with a subtle call to action

Write ONLY the marketing copy. No labels, no explanations."""

    return prompt


# ── Test prompts ──────────────────────────────────────────
sample_product  = "Sony WH-1000XM5 Noise Cancelling Headphones 30hr battery"
sample_category = "Electronics"
sample_result   = classify_product(sample_product, sample_category)

print("\n── GENERATED PROMPTS ────────────────────────────────")
print("\n[EMOTIONAL PROMPT]")
print(build_emotional_prompt(sample_product, sample_category,
                             sample_result['confidence']))

print("\n[RATIONAL PROMPT]")
print(build_rational_prompt(sample_product, sample_category,
                            sample_result['confidence']))


── GENERATED PROMPTS ────────────────────────────────

[EMOTIONAL PROMPT]
You are an expert neuro-marketing copywriter specializing in System 1 emotional marketing. System 1 thinking is fast, intuitive, and emotion-driven.

PRODUCT: Sony WH-1000XM5 Noise Cancelling Headphones 30hr battery
CATEGORY: Electronics
EMOTIONAL INTENSITY NEEDED: very strong

Write a SHORT marketing message (2-3 sentences, max 60 words) that:
1. Triggers immediate emotional desire using sensory or emotional language
2. Creates a feeling of pleasure, excitement, or belonging
3. Uses simple, vivid words that bypass rational thinking
4. Does NOT mention technical specifications or rational justifications
5. Use vivid sensory language, excitement, and desire. Make it irresistible.

Write ONLY the marketing copy. No labels, no explanations.

[RATIONAL PROMPT]
You are an expert neuro-marketing copywriter specializing in System 2 rational marketing. System 2 thinking is slow, analytical, and evidence-based.

PRODUCT:

In [ ]:
import os
import time
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv("../config/.env")

# ── The only two differences from standard OpenAI ────────
# 1. base_url points to xAI instead of OpenAI
# 2. api_key uses your XAI_API_KEY

client = OpenAI(
    api_key  = os.getenv("XAI_API_KEY"),
    base_url = "https://api.x.ai/v1"        # xAI endpoint
)

# ── Choose your model ────────────────────────────────────
# grok-3          → balanced, good quality, lower cost (recommended for you)
# grok-3-fast     → faster responses, slightly lower quality
# grok-4.3        → most capable, higher cost
GROK_MODEL = "grok-3"

print(f"Grok client ready | Model: {GROK_MODEL}")
print(f"API key loaded: {'YES' if os.getenv('XAI_API_KEY') else 'NO — check your .env file'}")

In [ ]:

def generate_with_grok(prompt, max_tokens=150, temperature=0.75, retries=3):
    """
    Call Grok to generate marketing copy.

    Args:
        prompt      : str  — the full prompt from your prompt builder
        max_tokens  : int  — max length of generated copy (150 = ~2-3 sentences)
        temperature : float — 0.0=deterministic, 1.0=very creative. 0.75 is ideal
        retries     : int  — how many times to retry on failure

    Returns:
        str — generated marketing copy, or None if all retries fail
    """

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model    = GROK_MODEL,
                messages = [
                    {
                        "role":    "system",
                        "content": (
                            "You are a world-class marketing copywriter with deep "
                            "expertise in consumer psychology and neuro-marketing. "
                            "You always write concise, powerful, psychologically "
                            "targeted copy. Never add explanations or labels — "
                            "write only the marketing copy itself."
                        )
                    },
                    {
                        "role":    "user",
                        "content": prompt
                    }
                ],
                max_tokens  = max_tokens,
                temperature = temperature,
            )

            # Extract the text from the response
            copy_text = response.choices[0].message.content.strip()

            # Basic quality check — reject if too short or empty
            if len(copy_text.split()) < 8:
                print(f"  Warning: Generated copy too short ({len(copy_text.split())} words). Retrying...")
                continue

            return copy_text

        except Exception as e:
            error_type = type(e).__name__
            print(f"  Attempt {attempt+1}/{retries} failed: {error_type}: {e}")

            # Rate limit — wait longer before retry
            if "rate" in str(e).lower() or "429" in str(e):
                wait = 30 * (attempt + 1)
                print(f"  Rate limit hit. Waiting {wait}s...")
                time.sleep(wait)
            # Auth error — no point retrying
            elif "auth" in str(e).lower() or "401" in str(e):
                print("  Authentication failed. Check your XAI_API_KEY in config/.env")
                return None
            else:
                time.sleep(3)

    print(f"  All {retries} attempts failed.")
    return None


# ── Wrapper that matches the interface from Step 2B ──────
def generate_copy(prompt):
    """Single entry point for copy generation — used by the agent"""
    return generate_with_grok(prompt)

In [ ]:

# CONNECTION TEST — run this before anything else
def test_grok_connection():
    print("Testing Grok API connection...")
    print("-" * 45)

    test_prompt = """Write one sentence of emotional marketing copy 
                     for a chocolate bar. Write ONLY the copy."""

    result = generate_with_grok(test_prompt, max_tokens=60)

    if result:
        print(f"✓ Connection successful")
        print(f"  Model    : {GROK_MODEL}")
        print(f"  Response : {result}")
    else:
        print("✗ Connection failed — check your API key and internet connection")

    print("-" * 45)

test_grok_connection()

In [ ]:

# full piplelinw classifier + prompt + Grok


test_products = [
    ("Neutrogena Hydro Boost Water Gel moisturizer",             "Beauty"),
    ("Sony WH-1000XM5 Wireless Headphones 30hr battery",         "Electronics"),
    ("Haribo Gold-Bears Gummy Candy 5lb Party Bag",              "Grocery"),
]

print("=" * 60)
print("  FULL PIPELINE TEST: Classifier → Prompt → Grok")
print("=" * 60)

for product, category in test_products:
    print(f"\nProduct  : {product}")
    print(f"Category : {category}")

    # Step 2A — Classify
    result = classify_product(product, category)
    mode   = result['cognitive_mode']
    conf   = result['confidence']
    print(f"Mode     : {mode} (confidence: {conf:.2%})")

    # Step 2B — Build prompts
    emo_prompt = build_emotional_prompt(product, category, conf)
    rat_prompt = build_rational_prompt(product, category, conf)

    # Step 2C — Generate with Grok
    print("Generating emotional copy...")
    emotional_copy = generate_copy(emo_prompt)
    time.sleep(1)   # Small pause between calls

    print("Generating rational copy...")
    rational_copy = generate_copy(rat_prompt)
    time.sleep(1)

    print(f"\n  [EMOTIONAL] {emotional_copy}")
    print(f"  [RATIONAL]  {rational_copy}")
    print(f"  [RECOMMENDED] {'EMOTIONAL' if mode == 'System1' else 'RATIONAL'} copy")
    print("-" * 60)

In [ ]:
import json
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

In [ ]:
def dual_system_agent(product_text, category="unknown"):
    """
    Main agent function — orchestrates the full pipeline:
    1. Classify product cognitive mode
    2. Generate emotional copy
    3. Generate rational copy
    4. Evaluate copy quality
    5. Select recommended strategy
    6. Return structured JSON output for Components 2, 3, 4

    Args:
        product_text : str — product title + description
        category     : str — product category

    Returns:
        dict — complete agent output as JSON-ready dictionary
    """

    print(f"\nProcessing: {product_text[:60]}...")

    #  Classify
    classification = classify_product(product_text, category)
    mode       = classification['cognitive_mode']
    confidence = classification['confidence']
    s1_prob    = classification['s1_probability']
    s2_prob    = classification['s2_probability']

    print(f"  Classification: {mode} (confidence: {confidence:.2%})")

    # Build prompts 
    # Always generate BOTH copies regardless of classification
    # This gives your paper the A/B comparison data

    if confidence < 0.65:
        # Borderline product — use hybrid prompt as primary
        emotional_prompt = build_hybrid_prompt(product_text, category, s1_prob, s2_prob)
        rational_prompt  = build_rational_prompt(product_text, category, confidence)
        copy_type        = "hybrid"
    else:
        emotional_prompt = build_emotional_prompt(product_text, category, confidence)
        rational_prompt  = build_rational_prompt(product_text, category, confidence)
        copy_type        = "standard"

    # Generate copies 
    print("  Generating emotional copy...")
    emotional_copy = generate_copy(emotional_prompt)
    time.sleep(1)   # Avoid rate limits

    print("  Generating rational copy...")
    rational_copy  = generate_copy(rational_prompt)
    time.sleep(1)

    if not emotional_copy or not rational_copy:
        print("  WARNING: Generation failed. Check LLM connection.")
        return None

    #Evaluate copy quality 
    def evaluate_copy(text, expected_mode):
        """Score how well the copy matches the intended cognitive mode"""
        sentiment = sia.polarity_scores(text)
        words     = text.lower().split()

        s1_markers = ['feel', 'love', 'amazing', 'perfect', 'beautiful',
                      'enjoy', 'experience', 'dream', 'wonderful', '!']
        s2_markers = ['features', 'performance', 'quality', 'reliable',
                      'efficient', 'proven', 'compare', 'specifications',
                      'battery', 'compatible', 'warranty', 'technology']

        s1_hits = sum(1 for w in s1_markers if w in words)
        s2_hits = sum(1 for w in s2_markers if w in words)

        if expected_mode == "emotional":
            alignment = min(s1_hits / max(s1_hits + s2_hits, 1), 1.0)
        else:
            alignment = min(s2_hits / max(s1_hits + s2_hits, 1), 1.0)

        return {
            "sentiment_compound": round(sentiment['compound'], 4),
            "sentiment_positive": round(sentiment['pos'], 4),
            "word_count":         len(words),
            "mode_alignment":     round(alignment, 4),  # 0–1, higher = better match
            "s1_marker_hits":     s1_hits,
            "s2_marker_hits":     s2_hits
        }

    emotional_quality = evaluate_copy(emotional_copy, "emotional")
    rational_quality  = evaluate_copy(rational_copy,  "rational")

    # Select recommended strategy 
    # Rule: follow the classifier unless confidence is very low
    if confidence >= 0.65:
        recommended_strategy = "emotional" if mode == "System1" else "rational"
        recommended_copy     = emotional_copy if mode == "System1" else rational_copy
    else:
        # Low confidence — recommend the one with better alignment score
        if emotional_quality['mode_alignment'] >= rational_quality['mode_alignment']:
            recommended_strategy = "emotional"
            recommended_copy     = emotional_copy
        else:
            recommended_strategy = "rational"
            recommended_copy     = rational_copy

    # ── Stage 6: Build JSON output for other components ──
    output = {
        "input": {
            "product_text": product_text,
            "category":     category
        },
        "classification": {
            "cognitive_mode":  mode,
            "label":           classification['label'],
            "confidence":      confidence,
            "s1_probability":  s1_prob,
            "s2_probability":  s2_prob,
            "copy_type":       copy_type,
            "reasoning":       classification['reasoning']
        },
        "generated_copy": {
            "emotional": {
                "text":    emotional_copy,
                "quality": emotional_quality
            },
            "rational": {
                "text":    rational_copy,
                "quality": rational_quality
            }
        },
        "recommendation": {
            "strategy":      recommended_strategy,
            "selected_copy": recommended_copy,
            "explanation":   (
                f"Product classified as {mode} with {confidence:.0%} confidence. "
                f"{recommended_strategy.capitalize()} strategy recommended based on "
                f"{'classifier decision' if confidence >= 0.65 else 'copy alignment score'}."
            )
        },
        # This is what Components 2, 3, 4 actually receive
        "agent_output": {
            "cognitive_mode":    mode,
            "confidence":        confidence,
            "strategy":          recommended_strategy,
            "emotional_copy":    emotional_copy,
            "rational_copy":     rational_copy,
            "recommended_copy":  recommended_copy
        }
    }

    print(f"  Strategy: {recommended_strategy} copy recommended")
    return output


# ── Run agent on test products ────────────────────────────
test_products = [
    ("Neutrogena Hydro Boost Water Gel Face Moisturizer",               "Beauty"),
    ("Sony WH-1000XM5 Wireless Headphones 30 hour battery life",        "Electronics"),
    ("Haribo Gold-Bears Gummy Candy Original 5lb Party Bag",            "Grocery"),
    ("Garmin Forerunner 955 Solar GPS Running Smartwatch",              "Sports"),
    ("Fisher-Price Laugh Learn Baby Learning Toy Gift Set",             "Baby"),
    ("Instant Pot Duo 7-in-1 Electric Pressure Cooker 6 Quart",        "Kitchen"),
    ("Levi's Men's 511 Slim Fit Stretch Jeans",                        "Apparel"),
    ("Fujifilm Instax Mini 11 Instant Camera Blush Pink",              "Electronics"),
]

results = []
for product, category in test_products:
    output = dual_system_agent(product, category)
    if output:
        results.append(output)

print(f"\nGenerated outputs for {len(results)} products")

In [ ]:

# Save full JSON output
with open("../outputs/agent_outputs.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved → ../outputs/agent_outputs.json")

# Build flat CSV for analysis and paper
rows = []
for r in results:
    rows.append({
        "product":               r['input']['product_text'],
        "category":              r['input']['category'],
        "cognitive_mode":        r['classification']['cognitive_mode'],
        "confidence":            r['classification']['confidence'],
        "s1_probability":        r['classification']['s1_probability'],
        "s2_probability":        r['classification']['s2_probability'],
        "emotional_copy":        r['generated_copy']['emotional']['text'],
        "rational_copy":         r['generated_copy']['rational']['text'],
        "emo_sentiment":         r['generated_copy']['emotional']['quality']['sentiment_compound'],
        "rat_sentiment":         r['generated_copy']['rational']['quality']['sentiment_compound'],
        "emo_alignment":         r['generated_copy']['emotional']['quality']['mode_alignment'],
        "rat_alignment":         r['generated_copy']['rational']['quality']['mode_alignment'],
        "recommended_strategy":  r['recommendation']['strategy'],
        "recommended_copy":      r['recommendation']['selected_copy'],
    })

df_results = pd.DataFrame(rows)
df_results.to_csv("../outputs/generated_copy_examples.csv", index=False)
print("Saved → ../outputs/generated_copy_examples.csv")

# ── Copy Quality Evaluation Summary ──────────────────────
print("\n" + "="*60)
print("   COPY QUALITY EVALUATION REPORT")
print("="*60)

# Sentiment check: emotional copy should be MORE positive
avg_emo_sent = df_results['emo_sentiment'].mean()
avg_rat_sent = df_results['rat_sentiment'].mean()
print(f"\n[1] SENTIMENT GAP")
print(f"    Emotional copy avg sentiment : {avg_emo_sent:.4f}")
print(f"    Rational copy avg sentiment  : {avg_rat_sent:.4f}")
print(f"    Gap                          : {avg_emo_sent - avg_rat_sent:.4f}")
print(f"    STATUS → {'PASS' if avg_emo_sent > avg_rat_sent else 'FAIL'}")

# Alignment check: each copy should match its intended mode
avg_emo_align = df_results['emo_alignment'].mean()
avg_rat_align = df_results['rat_alignment'].mean()
print(f"\n[2] MODE ALIGNMENT")
print(f"    Emotional copy alignment : {avg_emo_align:.4f}")
print(f"    Rational copy alignment  : {avg_rat_align:.4f}")
print(f"    STATUS → {'PASS' if avg_emo_align > 0.4 and avg_rat_align > 0.4 else 'NEEDS IMPROVEMENT'}")

# Classification confidence distribution
print(f"\n[3] CLASSIFICATION CONFIDENCE")
print(f"    Average confidence : {df_results['confidence'].mean():.4f}")
print(f"    High (>0.85)       : {(df_results.confidence > 0.85).sum()} products")
print(f"    Medium (0.65–0.85) : {((df_results.confidence > 0.65) & (df_results.confidence <= 0.85)).sum()} products")
print(f"    Low (<0.65)        : {(df_results.confidence < 0.65).sum()} products")

# Strategy distribution
print(f"\n[4] STRATEGY DISTRIBUTION")
print(df_results['recommended_strategy'].value_counts().to_string())

print(f"\n[5] SAMPLE OUTPUT")
sample = df_results.iloc[0]
print(f"\n  Product   : {sample['product']}")
print(f"  Mode      : {sample['cognitive_mode']} ({sample['confidence']:.0%} confident)")
print(f"  Emotional : {sample['emotional_copy']}")
print(f"  Rational  : {sample['rational_copy']}")
print(f"  Recommended: {sample['recommended_strategy'].upper()} copy")

print("\n" + "="*60)
print("   STEP 2 COMPLETE")
print("   Next: Build FastAPI backend (api/main.py)")
print("="*60)

In [ ]:

import sys
sys.path.append("..")   # So notebook can find src/

from src.agent import DualSystemAgent

# Load the agent
agent = DualSystemAgent(model_path="models/roberta_checkpoint")

# Quick single product test
result = agent.run("Neutrogena Hydro Boost Face Moisturizer", "Beauty")

print("agent_output (what Components 2,3,4 receive):")
print(result["agent_output"])